# 03 Feature Engineering

Build transaction behavior features without target leakage, legacy-rule shortcutting, or high-cardinality account-ID encoding.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"D:\Project\Data Science\financial_fraud_detection_&_transaction_intelligence")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import pandas as pd

from src.config import PROCESSED_DATA_DIR
from src.features import build_features, fit_feature_params

In [3]:
train_sample = pd.read_csv(PROCESSED_DATA_DIR / "modeling_train_sample.csv")
params = fit_feature_params(train_sample)
engineered, feature_columns = build_features(train_sample.head(5000), params=params, include_flagged=False)

{
    "feature_count": len(feature_columns),
    "target_in_features": "isFraud" in feature_columns,
    "legacy_flag_in_final_features": "isFlaggedFraud" in feature_columns,
    "account_id_in_features": bool({"nameOrig", "nameDest"} & set(feature_columns)),
    "composite_rule_score_in_model": "risk_rule_score" in feature_columns,
}

{'feature_count': 26,
 'target_in_features': False,
 'legacy_flag_in_final_features': False,
 'account_id_in_features': False,
 'composite_rule_score_in_model': False}

In [4]:
engineered[
    [
        "type",
        "amount",
        "amount_log",
        "origin_balance_delta",
        "origin_balance_error",
        "is_origin_account_drained",
        "amount_equals_old_origin_balance",
        "risk_rule_score",
        "fraud_risk_segment",
    ]
].head(12)

,type,amount,amount_log,origin_balance_delta,origin_balance_error,is_origin_account_drained,amount_equals_old_origin_balance,risk_rule_score,fraud_risk_segment
0,CASH_OUT,665441.98,13.408208,645.00,664796.98,1,0,82,Critical
1,CASH_OUT,135210.28,11.814594,135210.28,0.00,0,0,28,Medium
2,CASH_IN,378975.04,12.845228,-378975.04,757950.08,0,0,8,Low
3,CASH_IN,95482.05,11.466704,-95482.05,190964.10,0,0,8,Low
4,CASH_OUT,214952.27,12.278176,895.00,214057.27,1,0,70,High
5,CASH_IN,295804.57,12.597458,-295804.57,591609.14,0,0,8,Low
6,PAYMENT,27575.01,10.224701,5524.00,22051.01,1,0,42,Medium
7,CASH_IN,285138.54,12.560734,-285138.54,570277.08,0,0,18,Low
8,CASH_IN,27666.29,10.228006,-27666.29,55332.58,0,0,18,Low
9,CASH_OUT,76950.92,11.250936,0.00,76950.92,0,0,36,Medium


`risk_rule_score` is kept as a business segmentation signal, but excluded from the final ML feature set so model interpretation remains focused on original transaction behavior.